# Berlin Urban Flood — Phase 3: Full 13-feature SA1 Random Forest pilot

**Purpose:** Train/evaluate a **real-data Random Forest** with all **12 published spatial features + rainfall** on SA1, without extracting ~59 GB of rasters simultaneously.

This is an **independent pilot, NOT a complete reproduction of Seleem et al. (2023)**: uses a small grid of spatially separated tiles and six selected precipitation scenarios. It does **not** reproduce the paper's 10%-of-all-pixels sampling, SA1&SA2 training, or spatial-domain transfer to SA0.

**Important:** Full mode must transfer **~20.8 GB compressed** spatial features and crop 12 large GeoTIFF files one at a time. It can be slow, may exceed Kaggle session limits and depends on HTTP Range availability. If Kaggle interrupts, partial crops saved in `/kaggle/working` can be reused **only if preserved via a notebook version/dataset**. No accuracy is guaranteed. This notebook has been **syntax-checked but not run on Kaggle**.

[Original paper](https://doi.org/10.5194/nhess-23-809-2023) · [Dataset used here](https://zenodo.org/records/7221058) · [Original code](https://github.com/omarseleem92/Machine_learning_for_urban_flooding)

## 1 — Settings and hardware

Use **Internet: On**. GPU is **not required** for Random Forest: select Accelerator **None**. Avoid `Save & Run All` until all the cells work on the first pass.

In [3]:
import os, sys, subprocess, importlib.util, time, json, shutil, zlib, struct
from pathlib import Path
from collections import defaultdict

import numpy as np
RECORD='7221058'
OUT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/tmp/berlin_phase3_output')
TEMP=Path('/kaggle/tmp/berlin_phase3') if Path('/kaggle').is_dir() else Path('/tmp/berlin_phase3')
OUT.mkdir(parents=True,exist_ok=True)
TEMP.mkdir(parents=True,exist_ok=True)
CROPS=OUT/'berlin_sa1_aligned_predictors'
CROPS.mkdir(exist_ok=True)
SEED=42
TRAIN_RAIN=[40,60,80,120]
VAL_RAIN=90
TEST_RAIN=100 # Reserved paper testing event; DO NOT tune hyperparameters on these results.
N_TILES_AXIS=6
N_PIXELS_PER_TILE_EVENT=700
RF_TREES=10
print('Python:',sys.version.split()[0]); print('Output:',OUT); print('Scratch:',TEMP)
print('Scratch free GiB:',round(shutil.disk_usage(TEMP).free/2**30,2))
print('Working free GiB:',round(shutil.disk_usage(OUT).free/2**30,2))


Python: 3.13.15
Output: /kaggle/working
Scratch: /kaggle/tmp/berlin_phase3
Scratch free GiB: 1048.77
Working free GiB: 19.5


## 2 — Install imports and inspect available ZIP files

We reuse the working **Deflate64/Zstandard HTTP Range** extraction approach from the corrected Phase 2 notebook.

In [4]:
packages={'remotezip':'remotezip','rasterio':'rasterio','scikit-learn':'sklearn',
          'pandas':'pandas','matplotlib':'matplotlib','requests':'requests',
          'inflate64':'inflate64','zstandard':'zstandard','joblib':'joblib'}
missing=[pkg for pkg,mod in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing:',missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import requests, rasterio, pandas as pd, joblib, sklearn
from remotezip import RemoteZip
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
print('rasterio:',rasterio.__version__, 'scikit-learn:',sklearn.__version__)


Installing: ['remotezip', 'inflate64']
rasterio: 1.5.1 scikit-learn: 1.6.1


In [5]:
FEATURES=[
    'CN.tif', 'Cos_Aspect.tif', 'Curvature.tif',
    'DEM_berlin_boundary.tif','DEM_depth_1m_clip.tif', 'DEM_L.tif',
    'flow_acc_comb_nodata_clipped.tif', 'roughness_final_berlin.tif',
    'Sin_Aspect.tif', 'Slope_degree.tif', 'TPI_nan.tif',
    'TWI_Low_pass_filter_berlin_boundary.tif'
]
RAIN_SCENARIOS=sorted(set(TRAIN_RAIN+[VAL_RAIN,TEST_RAIN]))
ZIP_METHOD={0:'Stored',8:'Deflate',9:'Deflate64',12:'BZIP2',14:'ZIP LZMA',20:'Zstd',93:'Zstd'}
def zip_url(archive):
    return f'https://zenodo.org/records/{RECORD}/files/{archive}?download=1'
for archive,needed in [('Predictive_features.zip',FEATURES),
                       ('Water_depth_TELEMAC_2D.zip',[f'SA1/WD_{r}mm.tif' for r in RAIN_SCENARIOS])]:
    with RemoteZip(zip_url(archive),timeout=180) as remote:
        infos={info.filename:info for info in remote.infolist()}
        missing_names=set(needed)-set(infos)
        assert not missing_names, f'Missing in {archive}: {missing_names}'
        print(archive,':',len(needed),'selected files; total selected compressed GB:',
              round(sum(infos[n].compress_size for n in needed)/1e9,3))
        for name in needed:
            info=infos[name]
            print(name,'method',info.compress_type,ZIP_METHOD.get(info.compress_type,'UNKNOWN'),
                  'compressed GB',round(info.compress_size/1e9,3),
                  'unpacked GB',round(info.file_size/1e9,3))


Predictive_features.zip : 12 selected files; total selected compressed GB: 20.764
CN.tif method 9 Deflate64 compressed GB 0.012 unpacked GB 0.032
Cos_Aspect.tif method 9 Deflate64 compressed GB 2.943 unpacked GB 6.926
Curvature.tif method 9 Deflate64 compressed GB 2.508 unpacked GB 6.925
DEM_berlin_boundary.tif method 9 Deflate64 compressed GB 1.284 unpacked GB 1.312
DEM_depth_1m_clip.tif method 9 Deflate64 compressed GB 0.488 unpacked GB 0.516
DEM_L.tif method 9 Deflate64 compressed GB 1.661 unpacked GB 6.926
flow_acc_comb_nodata_clipped.tif method 9 Deflate64 compressed GB 0.945 unpacked GB 0.977
roughness_final_berlin.tif method 9 Deflate64 compressed GB 0.033 unpacked GB 7.077
Sin_Aspect.tif method 9 Deflate64 compressed GB 2.943 unpacked GB 6.926
Slope_degree.tif method 9 Deflate64 compressed GB 2.896 unpacked GB 6.926
TPI_nan.tif method 9 Deflate64 compressed GB 1.41 unpacked GB 6.925
TWI_Low_pass_filter_berlin_boundary.tif method 9 Deflate64 compressed GB 3.639 unpacked GB 3.841

## 3 — Shared HTTP Range ZIP reader

Never store the whole archive. The reader downloads only the compressed bytes of the requested member, checks exact length and CRC, and deletes incomplete files on failure.

In [6]:
import inflate64
import zstandard as zstd
RANGE_CHUNK=16*2**20

def fetch_range(url,start,length,session):
    if length==0:return b''
    end=start+length-1
    headers={'Range':f'bytes={start}-{end}','Accept-Encoding':'identity'}
    for attempt in range(3):
        try:
            with session.get(url,headers=headers,stream=True,timeout=(30,180)) as response:
                if response.status_code!=206:
                    raise RuntimeError(f'HTTP Range not honored: HTTP {response.status_code} -- STOP')
                actual=response.headers.get('Content-Range','')
                if not actual.startswith(f'bytes {start}-{end}/'):
                    raise RuntimeError('Unexpected Content-Range: '+actual)
                data=bytearray()
                for part in response.iter_content(chunk_size=2**20):
                    data.extend(part)
                    if len(data)>length:raise RuntimeError('Unexpected excess HTTP bytes')
            if len(data)!=length:raise IOError(f'Incomplete HTTP Range {len(data)} != {length}')
            return bytes(data)
        except (requests.exceptions.RequestException,IOError) as exc:
            if attempt==2:raise
            print('  Retrying range after:',type(exc).__name__)
            time.sleep(2**attempt)

def payload_offset(url,info,session):
    header=fetch_range(url,info.header_offset,30,session)
    (sig,ver,flags,method,mtime,mdate,crc,comp32,raw32,nlen,elen)=struct.unpack('<IHHHHHIIIHH',header)
    if sig!=0x04034b50 or method!=info.compress_type or (flags&1):
        raise RuntimeError('Unsupported encrypted or inconsistent ZIP member')
    return info.header_offset+30+nlen+elen

def extract_one(archive,name):
    filename=TEMP/archive.replace('.zip','')/name
    filename.parent.mkdir(parents=True,exist_ok=True)
    url=zip_url(archive)
    with RemoteZip(url,timeout=180) as remote:
        info=remote.getinfo(name)
        if filename.is_file() and filename.stat().st_size==info.file_size:
            print('Scratch cache:',name)
            return filename
        if shutil.disk_usage(TEMP).free < info.file_size+2*2**30:
            raise RuntimeError(f'Insufficient scratch disk for {name}: need at least '
                               f'{(info.file_size+2*2**30)/2**30:.1f} GiB free')
        tmp=filename.with_suffix(filename.suffix+'.part')
        if tmp.exists():tmp.unlink()
        crc=0;n=0;t=time.time()
        print('EXTRACT',name,'compressed GB',round(info.compress_size/1e9,2),
              'uncompressed GB',round(info.file_size/1e9,2))
        try:
            if info.compress_type in (0,8,12,14):
                with remote.open(info) as src,tmp.open('wb') as out:
                    while True:
                        data=src.read(2**20)
                        if not data:break
                        out.write(data); n+=len(data);crc=zlib.crc32(data,crc)
            elif info.compress_type in (9,20,93):
                if info.compress_type==9:
                    dec=inflate64.Inflater();decode=dec.inflate;flush=lambda:b''
                else:
                    dec=zstd.ZstdDecompressor().decompressobj();decode=dec.decompress;flush=dec.flush
                with requests.Session() as session:
                    offset=payload_offset(url,info,session)
                    with tmp.open('wb') as out:
                        for pos in range(0,info.compress_size,RANGE_CHUNK):
                            size=min(RANGE_CHUNK,info.compress_size-pos)
                            data=decode(fetch_range(url,offset+pos,size,session))
                            if data:out.write(data);n+=len(data);crc=zlib.crc32(data,crc)
                            if (pos//RANGE_CHUNK)%16==0:
                                print('  Compressed downloaded',round(min(pos+size,info.compress_size)/1e9,2),
                                      '/',round(info.compress_size/1e9,2),'GB')
                        tail=flush()
                        if tail:out.write(tail);n+=len(tail);crc=zlib.crc32(tail,crc)
            else:raise NotImplementedError('ZIP compression method: '+str(info.compress_type))
            if n!=info.file_size or (crc&0xffffffff)!=info.CRC:
                raise IOError(f'ZIP size/CRC mismatch for {name}: {n} bytes')
            tmp.replace(filename)
            print('CRC OK:',name,'seconds',round(time.time()-t,1))
        finally:
            if tmp.exists():tmp.unlink()
    return filename


In [8]:
import time

# Preserve the original function.
if "_fetch_range_original" not in globals():
    _fetch_range_original = fetch_range

def fetch_range(url, start, length, session):
    max_attempts = 8

    for attempt in range(max_attempts):
        try:
            return _fetch_range_original(
                url, start, length, session
            )

        except RuntimeError as e:
            error = str(e)

            retryable = any(
                f"HTTP {code}" in error
                for code in [429, 500, 502, 503, 504]
            )

            if not retryable:
                raise

            if attempt == max_attempts - 1:
                raise

            delay = min(60, 5 * (2 ** attempt))

            print(
                f"Server temporarily unavailable: {error}"
            )
            print(
                f"Retry {attempt + 2}/{max_attempts} "
                f"in {delay} seconds..."
            )

            time.sleep(delay)

print("HTTP retry patch installed successfully!")

HTTP retry patch installed successfully!


## 4 — Establish SA1 reference grid, then crop predictors **one at a time**

**Run this cell carefully.** It transfers approximately **20.8 GB compressed** in total and may take significant time. At no point should all original 12 Berlin-wide raster files be extracted simultaneously. Each cropped raster is stored in `/kaggle/working/berlin_sa1_aligned_predictors/` and the huge scratch TIFF is deleted before processing the next one.

**Fallback if Kaggle is interrupted:** Check which `.tif` crops were saved in a completed notebook version. Don't assume in-session scratch survives restarts.

In [9]:
# Section 4 (fixed): resume from existing aligned crops and verify missing-CRS rasters.
# A missing CRS is NOT proof that the file uses EPSG:25833.
# Only assume GRID['crs'] if the raster already has plausible Berlin UTM coordinates,
# suitable pixel size, and covers at least 95% of the SA1 target bounding box.
from rasterio.transform import array_bounds

REFERENCE=extract_one('Water_depth_TELEMAC_2D.zip','SA1/WD_60mm.tif')
with rasterio.open(REFERENCE) as ds:
    GRID=dict(crs=ds.crs,transform=ds.transform,width=ds.width,height=ds.height)
    print('SA1 reference:',GRID['crs'],GRID['width'],GRID['height'],ds.res,ds.bounds)
    assert GRID['crs'] is not None

CATEGORICAL={'CN.tif','roughness_final_berlin.tif'}

def checked_source_crs(src, name):
    if src.crs is not None:
        return src.crs, False
    # Compare numerical coordinates, NOT geography: this tests whether assuming
    # the reference CRS is plausible. It does not prove the missing CRS.
    rleft,rbottom,rright,rtop=array_bounds(GRID['height'],GRID['width'],GRID['transform'])
    b=src.bounds
    overlap_w=max(0.0,min(b.right,rright)-max(b.left,rleft))
    overlap_h=max(0.0,min(b.top,rtop)-max(b.bottom,rbottom))
    overlap=overlap_w*overlap_h/((rright-rleft)*(rtop-rbottom))
    sx,sy=map(abs,src.res)
    plausible_res=(0.25<=sx<=10 and 0.25<=sy<=10)
    plausible_xy=(100_000<b.left<900_000 and 5_000_000<b.bottom<7_000_000)
    print(f'WARNING: {name} has NO CRS. Bounds={b}, resolution={src.res}, '
          f'coverage_if_assumed={overlap:.2%}')
    if not (overlap>=.95 and plausible_res and plausible_xy):
        raise RuntimeError(
            f'Missing CRS: {name}. Cannot safely assume {GRID["crs"]}. '
            'Inspect raw raster metadata before continuing.')
    print(f'PROVISIONAL ASSUMPTION: {name} uses {GRID["crs"]} '
          '(recorded in output metadata; must be verified before paper reproduction).')
    return GRID['crs'], True

report=[]
for fi,feature in enumerate(FEATURES,1):
    output=CROPS/(Path(feature).stem+'_SA1.tif')
    if output.is_file():
        with rasterio.open(output) as cached:
            valid_cache=(cached.width==GRID['width'] and cached.height==GRID['height']
                         and cached.crs==GRID['crs']
                         and cached.transform.almost_equals(GRID['transform']))
        if valid_cache:
            print(f'[{fi}/12] KEEP cached {feature} (no download)')
            report.append(dict(feature=feature,status='cached',file=str(output)))
            pd.DataFrame(report).to_csv(OUT/'berlin_phase3_crop_progress.csv',index=False)
            continue

    raw=extract_one('Predictive_features.zip',feature)
    succeeded=False
    try:
        with rasterio.open(raw) as src:
            source_crs,assumed=checked_source_crs(src,feature)
            mode=Resampling.nearest if feature in CATEGORICAL else Resampling.bilinear
            with WarpedVRT(src,src_crs=source_crs,crs=GRID['crs'],
                           transform=GRID['transform'],width=GRID['width'],
                           height=GRID['height'],resampling=mode,
                           dtype='float32',nodata=np.nan) as vrt:
                profile=dict(driver='GTiff',width=GRID['width'],height=GRID['height'],count=1,
                             dtype='float32',crs=GRID['crs'],transform=GRID['transform'],
                             nodata=np.nan,tiled=True,blockxsize=256,blockysize=256,
                             compress='DEFLATE',predictor=3)
                partial=output.with_suffix('.partial.tif')
                if partial.exists():partial.unlink()
                count=0;total=0
                with rasterio.open(partial,'w',**profile) as dst:
                    if assumed:
                        dst.update_tags(source_crs_assumed=str(source_crs),
                                        source_crs_original='missing',
                                        assumption='SA1 bounds overlap and pixel size verified')
                    for _,win in dst.block_windows(1):
                        block=vrt.read(1,window=win,out_dtype='float32',masked=True)
                        arr=np.asarray(block.filled(np.nan),dtype='float32')
                        count+=int(np.isfinite(arr).sum());total+=arr.size
                        dst.write(arr,1,window=win)
                if count<1000:
                    partial.unlink(missing_ok=True)
                    raise RuntimeError(f'Almost no valid pixels in aligned feature {feature}; inspect CRS/NoData')
                partial.replace(output)
                succeeded=True
                print(f'[{fi}/12] SAVED {feature}: valid={count}/{total} ({count/total:.1%})')
                report.append(dict(feature=feature,status='extracted',
                                   assumed_crs=assumed,valid_fraction=count/total,file=str(output)))
    finally:
        if succeeded:
            raw.unlink(missing_ok=True)
        else:
            print('Kept source raster for debugging / retry:',raw)
    pd.DataFrame(report).to_csv(OUT/'berlin_phase3_crop_progress.csv',index=False)
print('DONE. Number of aligned predictors:',len(list(CROPS.glob('*.tif'))))


Scratch cache: SA1/WD_60mm.tif
SA1 reference: EPSG:25833 3253 3577 (1.0000298493698168, 1.0001388873357138) BoundingBox(left=384570.6118, bottom=5823429.9831, right=387823.7089, top=5827007.4799)
[1/12] KEEP cached CN.tif (no download)
[2/12] KEEP cached Cos_Aspect.tif (no download)
[3/12] KEEP cached Curvature.tif (no download)
[4/12] KEEP cached DEM_berlin_boundary.tif (no download)
EXTRACT DEM_depth_1m_clip.tif compressed GB 0.49 uncompressed GB 0.52
  Compressed downloaded 0.02 / 0.49 GB
  Compressed downloaded 0.29 / 0.49 GB
CRC OK: DEM_depth_1m_clip.tif seconds 385.3
[5/12] SAVED DEM_depth_1m_clip.tif: valid=11635981/11635981 (100.0%)
EXTRACT DEM_L.tif compressed GB 1.66 uncompressed GB 6.93
  Compressed downloaded 0.02 / 1.66 GB
  Compressed downloaded 0.29 / 1.66 GB
  Compressed downloaded 0.55 / 1.66 GB
  Compressed downloaded 0.82 / 1.66 GB
  Compressed downloaded 1.09 / 1.66 GB
  Compressed downloaded 1.36 / 1.66 GB
  Compressed downloaded 1.63 / 1.66 GB
CRC OK: DEM_L.tif se

## 5 — Load six real rainfall targets (40, 60, 80, 90, 100, 120 mm)

Training rainfall scenarios are **40,60,80,120**. Validation uses **90 mm** on held-out geographic tiles. **100 mm** is a test-only scenario and is **never** used to tune the model. These are *pilot choices* and not the publication's exact split.

In [10]:
TARGETS={60:REFERENCE}
for rain in RAIN_SCENARIOS:
    if rain==60:continue
    TARGETS[rain]=extract_one('Water_depth_TELEMAC_2D.zip',f'SA1/WD_{rain}mm.tif')
for rain,path in sorted(TARGETS.items()):
    with rasterio.open(path) as ds:
        assert ds.crs is not None,f'CRS missing for rain {rain}'
        print(rain,'mm:',ds.width,ds.height,ds.crs,ds.bounds)


EXTRACT SA1/WD_40mm.tif compressed GB 0.08 uncompressed GB 0.09
CRC OK: SA1/WD_40mm.tif seconds 106.2
EXTRACT SA1/WD_80mm.tif compressed GB 0.08 uncompressed GB 0.09
CRC OK: SA1/WD_80mm.tif seconds 64.1
EXTRACT SA1/WD_90mm.tif compressed GB 0.08 uncompressed GB 0.09
CRC OK: SA1/WD_90mm.tif seconds 48.7
EXTRACT SA1/WD_100mm.tif compressed GB 0.08 uncompressed GB 0.09
CRC OK: SA1/WD_100mm.tif seconds 121.5
EXTRACT SA1/WD_120mm.tif compressed GB 0.08 uncompressed GB 0.09
CRC OK: SA1/WD_120mm.tif seconds 81.0
40 mm: 3253 3577 EPSG:25833 BoundingBox(left=384570.6118, bottom=5823429.9831, right=387823.7089, top=5827007.4799)
60 mm: 3253 3577 EPSG:25833 BoundingBox(left=384570.6118, bottom=5823429.9831, right=387823.7089, top=5827007.4799)
80 mm: 3253 3577 EPSG:25833 BoundingBox(left=384570.6118, bottom=5823429.9831, right=387823.7089, top=5827007.4799)
90 mm: 3253 3577 EPSG:25833 BoundingBox(left=384570.6118, bottom=5823429.9831, right=387823.7089, top=5827007.4799)
100 mm: 3253 3577 EPSG:25

## 6 — Sample spatially separated tiles and build a small training matrix

36 spatial tiles are divided into **27 training / 9 validation tiles** (seed 42). Training, validation and test use **disjoint locations** for this pilot; train, val and test also use **different rainfall scenarios**. We keep all 12 input features in the same order plus rainfall in millimeters.

This pilot uses only 700 randomly selected valid pixels per tile per rainfall, not all pixels.

In [11]:
from contextlib import ExitStack
with ExitStack() as stack:
    ds_list=[stack.enter_context(rasterio.open(CROPS/(Path(f).stem+'_SA1.tif'))) for f in FEATURES]
    with rasterio.open(REFERENCE) as ref:
        width,height=ref.width,ref.height
        assert all(d.width==width and d.height==height for d in ds_list)
        print('All 12 inputs aligned to:',width,'x',height)
    all_tiles=[]
    for iy in range(N_TILES_AXIS):
        for ix in range(N_TILES_AXIS):
            row0=min(int((iy+.5)*height/N_TILES_AXIS),max(0,height-256))
            col0=min(int((ix+.5)*width/N_TILES_AXIS),max(0,width-256))
            all_tiles.append((iy*N_TILES_AXIS+ix, Window(col0,row0,256,256)))

train_tile_ids,val_tile_ids=train_test_split([t for t,_ in all_tiles],test_size=.25,random_state=SEED)
train_tile_ids=set(train_tile_ids);val_tile_ids=set(val_tile_ids)
assert not train_tile_ids.intersection(val_tile_ids)
print('Train tiles:',sorted(train_tile_ids))
print('Validation/test tiles:',sorted(val_tile_ids))


All 12 inputs aligned to: 3253 x 3577
Train tiles: [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 14, 15, 17, 18, 19, 20, 22, 23, 24, 25, 27, 28, 29, 32, 33, 34]
Validation/test tiles: [8, 12, 13, 16, 21, 26, 30, 31, 35]


In [12]:
from contextlib import ExitStack
rng=np.random.default_rng(SEED)
buckets={part:[] for part in ['train','val','test']}
used_tiles=defaultdict(set)
with ExitStack() as stack:
    predictors=[stack.enter_context(rasterio.open(CROPS/(Path(f).stem+'_SA1.tif'))) for f in FEATURES]
    raw_target=[stack.enter_context(rasterio.open(TARGETS[r])) for r in RAIN_SCENARIOS]
    aligned_targets={r:stack.enter_context(WarpedVRT(ds,crs=GRID['crs'],
                                    transform=GRID['transform'],width=GRID['width'],height=GRID['height'],
                                    dtype='float32',nodata=np.nan,resampling=Resampling.nearest))
                     for r,ds in zip(RAIN_SCENARIOS,raw_target)}
    for tile_id,win in all_tiles:
        split='train' if tile_id in train_tile_ids else 'heldout'
        feature_arrays=[ds.read(1,window=win,masked=True).astype('float32').filled(np.nan)
                        for ds in predictors]
        features=np.stack(feature_arrays,axis=-1)
        finite_x=np.isfinite(features).all(axis=-1)
        for rain in RAIN_SCENARIOS:
            if split=='train' and rain not in TRAIN_RAIN:continue
            if split=='heldout' and rain not in [VAL_RAIN,TEST_RAIN]:continue
            part='train' if split=='train' else ('val' if rain==VAL_RAIN else 'test')
            y=aligned_targets[rain].read(1,window=win,masked=True).astype('float32').filled(np.nan)
            valid=finite_x & np.isfinite(y) & (y>=0)
            yy,xx=np.where(valid)
            if len(yy)<50:
                print('SKIP tile',tile_id,'rain',rain,'valid',len(yy));continue
            inds=rng.choice(len(yy),size=min(N_PIXELS_PER_TILE_EVENT,len(yy)),replace=False)
            rows,cols=yy[inds],xx[inds]
            x=np.column_stack([features[rows,cols,:],np.full(len(inds),rain,dtype='float32')]).astype('float32')
            targets=y[rows,cols].astype('float32')
            buckets[part].append((x,targets))
            used_tiles[part].add(tile_id)
        if tile_id%6==0:print('Sampled tile',tile_id)

SAMPLES={}
for part,blocks in buckets.items():
    if not blocks:raise RuntimeError(f'No valid examples in {part}; inspect NoData and raster alignment')
    xx=np.concatenate([x for x,y in blocks],axis=0)
    yy=np.concatenate([y for x,y in blocks],axis=0)
    SAMPLES[part]=(xx,yy)
    print(part,'X shape',xx.shape,'y range',float(yy.min()),float(yy.max()),
          'rain scenarios',sorted(np.unique(xx[:,-1]).astype(int).tolist()),
          'tiles',len(used_tiles[part]))
assert SAMPLES['train'][0].shape[-1]==13
assert used_tiles['train'].isdisjoint(used_tiles['val'])
assert used_tiles['train'].isdisjoint(used_tiles['test'])


Sampled tile 0
Sampled tile 6
Sampled tile 12
Sampled tile 18
Sampled tile 24
Sampled tile 30
train X shape (75600, 13) y range 9.995872005674755e-07 2.0108532905578613 rain scenarios [40, 60, 80, 120] tiles 27
val X shape (6300, 13) y range 9.998051382353879e-07 3.7738804817199707 rain scenarios [90] tiles 9
test X shape (6300, 13) y range 9.998236691899365e-07 4.598528861999512 rain scenarios [100] tiles 9


## 7 — Train two comparable RF models

Compare **2 spatial predictors + rainfall** versus **all 12 spatial predictors + rainfall** using the **same samples and same rainfall and spatial split**. Both use 10 trees as in the source RF notebook's initial setting (still a pilot due to sampling and SA1 scope). Do not inspect test results to tune parameters.

In [13]:
FEATURE_LABELS=[Path(f).stem for f in FEATURES]+['Rainfall_mm']
PILOT_FEATURES=[FEATURES.index('CN.tif'),FEATURES.index('DEM_depth_1m_clip.tif'),12]
X_train,y_train=SAMPLES['train']
X_val,y_val=SAMPLES['val']
X_test,y_test=SAMPLES['test']
MODELS={}

for label,cols in [('rf_2spatial_plus_rain',PILOT_FEATURES),
                   ('rf_12spatial_plus_rain',list(range(13)))]:
    print('Training:',label,'columns:',[FEATURE_LABELS[i] for i in cols])
    model=RandomForestRegressor(n_estimators=RF_TREES,random_state=SEED,n_jobs=4)
    t=time.time()
    model.fit(X_train[:,cols],y_train)
    print('Seconds:',round(time.time()-t,1))
    MODELS[label]=(model,cols)


Training: rf_2spatial_plus_rain columns: ['CN', 'DEM_depth_1m_clip', 'Rainfall_mm']
Seconds: 0.7
Training: rf_12spatial_plus_rain columns: ['CN', 'Cos_Aspect', 'Curvature', 'DEM_berlin_boundary', 'DEM_depth_1m_clip', 'DEM_L', 'flow_acc_comb_nodata_clipped', 'roughness_final_berlin', 'Sin_Aspect', 'Slope_degree', 'TPI_nan', 'TWI_Low_pass_filter_berlin_boundary', 'Rainfall_mm']
Seconds: 10.3


## 8 — Evaluate validation and held-out rainfall event

Reported metrics: RMSE, MAE, R²/NSE, critical success index (CSI) for depth **≥ 0.1 m**, and wet-pixel RMSE. A low all-pixel RMSE can hide large errors on wet pixels. Because 100 mm is a **held-out event**, do not optimize the model after inspecting its test score.

In [14]:
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

def evaluate(true,pred):
    wet=np.asarray(true)>=0.1
    forecast=np.asarray(pred)>=0.1
    tp=int(np.sum(wet&forecast));fp=int(np.sum(~wet&forecast));fn=int(np.sum(wet&~forecast))
    return dict(rmse=float(np.sqrt(mean_squared_error(true,pred))),
                mae=float(mean_absolute_error(true,pred)),
                r2_or_nse=float(r2_score(true,pred)) if np.var(true)>0 else None,
                csi_10cm=float(tp/(tp+fp+fn)) if (tp+fp+fn)>0 else None,
                wet_pixel_rmse=float(np.sqrt(np.mean((true[wet]-pred[wet])**2))) if wet.any() else None,
                wet_fraction=float(wet.mean()),
                n_samples=int(len(true)),
                y_true_max=float(np.max(true)),y_pred_max=float(np.max(pred)))
rows=[]
for label,(model,cols) in MODELS.items():
    for part,(xx,yy) in [('validation_90mm',SAMPLES['val']),('test_100mm',SAMPLES['test'])]:
        prediction=model.predict(xx[:,cols])
        item=dict(model=label,split=part,**evaluate(yy,prediction))
        rows.append(item)
        print(label,part,item)
results=pd.DataFrame(rows)
results.to_csv(OUT/'berlin_phase3_metrics.csv',index=False)


rf_2spatial_plus_rain validation_90mm {'model': 'rf_2spatial_plus_rain', 'split': 'validation_90mm', 'rmse': 0.15032456981624626, 'mae': 0.048942727113089064, 'r2_or_nse': 0.16039420533750137, 'csi_10cm': 0.26418152350081037, 'wet_pixel_rmse': 0.3890457917436632, 'wet_fraction': 0.1111111111111111, 'n_samples': 6300, 'y_true_max': 3.7738804817199707, 'y_pred_max': 1.4699447244405746}
rf_2spatial_plus_rain test_100mm {'model': 'rf_2spatial_plus_rain', 'split': 'test_100mm', 'rmse': 0.1568276532651329, 'mae': 0.0535569506012904, 'r2_or_nse': 0.10789579927111304, 'csi_10cm': 0.2794561933534743, 'wet_pixel_rmse': 0.3818222640341737, 'wet_fraction': 0.13063492063492063, 'n_samples': 6300, 'y_true_max': 4.598528861999512, 'y_pred_max': 1.4699447244405746}
rf_12spatial_plus_rain validation_90mm {'model': 'rf_12spatial_plus_rain', 'split': 'validation_90mm', 'rmse': 0.15279657196915578, 'mae': 0.05081316143778105, 'r2_or_nse': 0.13255347904691595, 'csi_10cm': 0.2544951590594744, 'wet_pixel_rms

In [15]:
for label,(model,cols) in MODELS.items():
    joblib.dump(dict(model=model,columns=[FEATURE_LABELS[i] for i in cols]),
                OUT/f'berlin_phase3_{label}.joblib',compress=3)
config={
    'status':'FULL_13_FEATURE_SA1_PILOT_NOT_PAPER_BASELINE',
    'data_record':RECORD,'source_paper':'10.5194/nhess-23-809-2023',
    'features':FEATURE_LABELS,'train_rain_mm':TRAIN_RAIN,'validation_rain_mm':VAL_RAIN,
    'test_rain_mm':TEST_RAIN,'spatial_train_tiles':sorted(train_tile_ids),
    'spatial_val_test_tiles':sorted(val_tile_ids),
    'per_tile_per_event':N_PIXELS_PER_TILE_EVENT,'n_trees':RF_TREES,
    'train_pixels':int(len(SAMPLES['train'][1])),
    'val_pixels':int(len(SAMPLES['val'][1])),'test_pixels':int(len(SAMPLES['test'][1])),
    'raster_target_units':'Assumed meters; verify original source metadata',
    'numpy':np.__version__,'rasterio':rasterio.__version__,'sklearn':sklearn.__version__}
(OUT/'berlin_phase3_experiment.json').write_text(json.dumps(config,indent=2))
print('OUTPUTS:')
for f in sorted(OUT.glob('berlin_phase3*')):
    if f.is_file():print(f.name,round(f.stat().st_size/2**20,2),'MiB')


OUTPUTS:
berlin_phase3_crop_progress.csv 0.0 MiB
berlin_phase3_experiment.json 0.0 MiB
berlin_phase3_metrics.csv 0.0 MiB
berlin_phase3_rf_12spatial_plus_rain.joblib 9.9 MiB
berlin_phase3_rf_2spatial_plus_rain.joblib 7.22 MiB


## 9 — Save artifacts and experiment manifest

Saving RF models is optional, but useful to reproduce the exact predictions later. Outputs are stored in `/kaggle/working`, unlike temporary full-sized TIFFs.

## 10 — Interpretation and next steps

1. Check that every one of the **12 predictors** has valid pixels and that `X` shape is `(..., 13)`.
2. Compare full model against the 2-feature + rainfall model on the **same validation** samples.
3. Report both RMSE and CSI/wet-region error. Never claim these SA1 results reproduce Figure 4 of the paper.
4. Next for **full paper reproduction**: generate training data across **SA1 + SA2**, reproduce the precipitation-event split and original sampling, and evaluate transfer to SA0, ideally using the published Zenodo record `7516408` after verifying dataset equivalence. Preserve code commit hashes, split definitions and preprocessing choices.

**Send to ChatGPT:** `berlin_phase3_crop_progress.csv`, `berlin_phase3_metrics.csv`, `berlin_phase3_experiment.json`, or the first failing traceback. `Crop_progress` is written incrementally: it is useful for debugging incomplete runs.

In [1]:
from pathlib import Path

p = Path("/kaggle/working/berlin_sa1_aligned_predictors")
t = Path("/kaggle/tmp/berlin_phase3/Water_depth_TELEMAC_2D/SA1")

print("Spatial features:", len(list(p.glob("*_SA1.tif"))))
print("Rainfall targets:", len(list(t.glob("WD_*mm.tif"))))

Spatial features: 0
Rainfall targets: 0


In [3]:

from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/tmp"),
    Path("/kaggle/input")
]

print("=== SEARCHING FOR BERLIN DATA ===")

for root in roots:
    print(f"\nFolder: {root}")
    print("Exists:", root.exists())

    if not root.exists():
        continue

    files = sorted(root.rglob("*.tif"))

    print("Total TIFF files:", len(files))

    for f in files[:30]:
        print(" ", f)

print("\n=== PHASE 3 RESULTS ===")

for name in [
    "berlin_phase3_metrics.csv",
    "berlin_phase3_experiment.json"
]:
    p = Path("/kaggle/working") / name
    print(name, "FOUND" if p.exists() else "MISSING")


=== SEARCHING FOR BERLIN DATA ===

Folder: /kaggle/working
Exists: True
Total TIFF files: 0

Folder: /kaggle/tmp
Exists: False

Folder: /kaggle/input
Exists: True
Total TIFF files: 0

=== PHASE 3 RESULTS ===
berlin_phase3_metrics.csv MISSING
berlin_phase3_experiment.json MISSING
